# 합성 데이터 생성

> 실제 데이터가 부족할 때 LLM을 활용해 **학습 데이터를 직접 만들어내는** 기법

이 노트북에서는 실제 API 호출 없이 **프롬프트 설계와 프로세스**를 학습합니다.  
실전에서는 GPT-4/Claude API를 연결하여 대량 생성합니다.

In [2]:
import json
import random
from collections import Counter

---
## 1. Self-Instruct: 시드에서 확장하기

Stanford Alpaca가 사용한 방식.  
소수의 **시드 데이터**를 few-shot으로 보여주고, LLM에게 새로운 예시를 생성시킨다.

In [3]:
# === Self-Instruct: 시드 데이터 준비 ===
# 시드 데이터는 직접 작성한 고품질 예시 5~10개
# 이 품질이 전체 합성 데이터의 품질을 결정한다

seed_data = [
    {
        "instruction": "Python에서 리스트의 중복 요소를 제거하는 방법을 설명하세요.",
        "output": "set()을 사용하면 중복이 자동 제거됩니다.\nlist(set([1,2,2,3])) -> [1,2,3]\n순서 유지가 필요하면 dict.fromkeys()를 사용합니다."
    },
    {
        "instruction": "HTTP GET과 POST의 차이점을 설명하세요.",
        "output": "GET: URL에 데이터 포함, 조회용, 캐싱 가능, 길이 제한 있음\nPOST: Body에 데이터 포함, 생성/수정용, 캐싱 안 됨, 길이 제한 없음"
    },
    {
        "instruction": "SQL에서 INNER JOIN과 LEFT JOIN의 차이를 설명하세요.",
        "output": "INNER JOIN: 양쪽 테이블 모두에 매칭되는 행만 반환\nLEFT JOIN: 왼쪽 테이블의 모든 행 + 오른쪽에서 매칭되는 행 반환 (매칭 없으면 NULL)"
    },
    {
        "instruction": "Git에서 merge와 rebase의 차이점은 무엇인가요?",
        "output": "merge: 두 브랜치를 합치며 merge commit 생성, 히스토리 보존\nrebase: 커밋을 다른 브랜치 위로 재배치, 히스토리가 깔끔해지지만 공유 브랜치에서는 위험"
    },
    {
        "instruction": "REST API에서 상태 코드 200, 400, 500의 의미를 설명하세요.",
        "output": "200 OK: 요청 성공\n400 Bad Request: 클라이언트 요청 오류 (잘못된 파라미터 등)\n500 Internal Server Error: 서버 내부 오류"
    },
]

print(f"시드 데이터: {len(seed_data)}개")
print("\n시드 내용:")
for i, s in enumerate(seed_data):
    inst = s['instruction'][:50]
    print(f"  [{i+1}] {inst}...")

시드 데이터: 5개

시드 내용:
  [1] Python에서 리스트의 중복 요소를 제거하는 방법을 설명하세요....
  [2] HTTP GET과 POST의 차이점을 설명하세요....
  [3] SQL에서 INNER JOIN과 LEFT JOIN의 차이를 설명하세요....
  [4] Git에서 merge와 rebase의 차이점은 무엇인가요?...
  [5] REST API에서 상태 코드 200, 400, 500의 의미를 설명하세요....


In [4]:
# === Self-Instruct 프롬프트 설계 ===
# LLM에게 시드 데이터를 few-shot으로 보여주고
# "이와 유사하지만 다른 주제의 예시를 생성하라"고 지시

def build_self_instruct_prompt(seeds, num_generate=3):
    """
    Self-Instruct용 프롬프트 생성.
    시드 중 랜덤 3개를 few-shot으로 보여줌 (매번 다른 조합 -> 다양성)
    """
    # 랜덤으로 3개 선택 (매 호출마다 다른 조합)
    selected = random.sample(seeds, min(3, len(seeds)))
    
    prompt = """당신은 고품질 프로그래밍 교육 데이터를 생성하는 전문가입니다.

아래는 프로그래밍 관련 instruction-output 예시입니다:

"""
    for i, s in enumerate(selected, 1):
        prompt += f"""예시 {i}:
instruction: {s['instruction']}
output: {s['output']}

"""
    
    prompt += f"""위 예시를 참고하여, 다음 규칙을 지켜 새로운 instruction-output 쌍을 {num_generate}개 생성하세요:

규칙:
1. 위 예시와 다른 주제여야 합니다
2. instruction은 명확하고 구체적이어야 합니다
3. output은 50자 이상, 실용적인 내용이어야 합니다
4. 한국어로 작성하세요
5. JSON 형식으로 출력하세요

출력 형식:
[{{"instruction": "...", "output": "..."}}]
"""
    return prompt

# 프롬프트 확인
prompt = build_self_instruct_prompt(seed_data)
print("Self-Instruct 프롬프트:")
print("=" * 60)
print(prompt[:800])
print("...")
print("\n-> 이 프롬프트를 GPT-4/Claude API에 보내면 새로운 데이터가 생성됨")
print("   매번 다른 시드 조합을 보여줘서 다양성을 확보")

Self-Instruct 프롬프트:
당신은 고품질 프로그래밍 교육 데이터를 생성하는 전문가입니다.

아래는 프로그래밍 관련 instruction-output 예시입니다:

예시 1:
instruction: Python에서 리스트의 중복 요소를 제거하는 방법을 설명하세요.
output: set()을 사용하면 중복이 자동 제거됩니다.
list(set([1,2,2,3])) -> [1,2,3]
순서 유지가 필요하면 dict.fromkeys()를 사용합니다.

예시 2:
instruction: Git에서 merge와 rebase의 차이점은 무엇인가요?
output: merge: 두 브랜치를 합치며 merge commit 생성, 히스토리 보존
rebase: 커밋을 다른 브랜치 위로 재배치, 히스토리가 깔끔해지지만 공유 브랜치에서는 위험

예시 3:
instruction: REST API에서 상태 코드 200, 400, 500의 의미를 설명하세요.
output: 200 OK: 요청 성공
400 Bad Request: 클라이언트 요청 오류 (잘못된 파라미터 등)
500 Internal Server Error: 서버 내부 오류

위 예시를 참고하여, 다음 규칙을 지켜 새로운 instruction-output 쌍을 3개 생성하세요:

규칙:
1. 위 예시와 다른 주제여야 합니다
2. instruction은 명확하고 구체적이어야 합니다
3. output은 50자 이상, 실용적인 내용이어야 합니다
4. 한국어로 작성하세요
5. JSON 형식으로 출력하세요

출력 형식:
[{"instruction": "...", "output": "..."}]

...

-> 이 프롬프트를 GPT-4/Claude API에 보내면 새로운 데이터가 생성됨
   매번 다른 시드 조합을 보여줘서 다양성을 확보


In [5]:
# === Self-Instruct 결과 시뮬레이션 ===
# 실제로는 LLM API를 호출하지만, 여기서는 프로세스 이해를 위해
# 미리 준비된 예시로 시뮬레이션

simulated_generated = [
    {
        "instruction": "Docker에서 컨테이너와 이미지의 차이를 설명하세요.",
        "output": "이미지: 실행 환경의 스냅샷 (읽기 전용 템플릿)\n컨테이너: 이미지를 실행한 인스턴스 (읽기/쓰기 가능)\n비유: 이미지=클래스, 컨테이너=인스턴스"
    },
    {
        "instruction": "Python에서 *args와 **kwargs의 차이점은?",
        "output": "*args: 위치 인자를 튜플로 받음. def f(*args) -> f(1,2,3)\n**kwargs: 키워드 인자를 딕셔너리로 받음. def f(**kwargs) -> f(a=1,b=2)\n둘 다 사용 시 *args가 먼저 와야 함"
    },
    {
        "instruction": "TCP와 UDP의 차이점을 설명하세요.",
        "output": "TCP: 연결 지향, 신뢰성 보장, 순서 보장, 느림 (웹, 이메일)\nUDP: 비연결, 신뢰성 미보장, 순서 미보장, 빠름 (스트리밍, 게임)"
    },
    {
        "instruction": "Python의 GIL(Global Interpreter Lock)이란 무엇인가요?",
        "output": "GIL은 한 번에 하나의 스레드만 Python 바이트코드를 실행하도록 제한하는 뮤텍스입니다.\nCPU-bound 작업에서 멀티스레딩 성능이 제한됩니다.\n해결: multiprocessing 사용 또는 C 확장"
    },
    {
        "instruction": "NoSQL 데이터베이스는 언제 사용하나요?",
        "output": "1. 스키마가 자주 변경되는 경우 (유연한 구조)\n2. 대량의 비정형 데이터 저장\n3. 수평 확장이 필요한 경우\n4. 실시간 빅데이터 처리\n대표: MongoDB(문서), Redis(키-값), Cassandra(컬럼)"
    },
]

print(f"생성된 데이터 (시뮬레이션): {len(simulated_generated)}개")
print("=" * 60)
for i, item in enumerate(simulated_generated):
    print(f"\n[{i+1}] {item['instruction']}")
    output_preview = item['output'][:80]
    print(f"    -> {output_preview}...")

print(f"\n-> 실전에서는 이 과정을 수백~수천 번 반복")
print(f"   시드 5개 -> 50개 -> 500개 -> 5000개로 확장")

생성된 데이터 (시뮬레이션): 5개

[1] Docker에서 컨테이너와 이미지의 차이를 설명하세요.
    -> 이미지: 실행 환경의 스냅샷 (읽기 전용 템플릿)
컨테이너: 이미지를 실행한 인스턴스 (읽기/쓰기 가능)
비유: 이미지=클래스, 컨테이너=인스턴...

[2] Python에서 *args와 **kwargs의 차이점은?
    -> *args: 위치 인자를 튜플로 받음. def f(*args) -> f(1,2,3)
**kwargs: 키워드 인자를 딕셔너리로 받음. def f...

[3] TCP와 UDP의 차이점을 설명하세요.
    -> TCP: 연결 지향, 신뢰성 보장, 순서 보장, 느림 (웹, 이메일)
UDP: 비연결, 신뢰성 미보장, 순서 미보장, 빠름 (스트리밍, 게임)...

[4] Python의 GIL(Global Interpreter Lock)이란 무엇인가요?
    -> GIL은 한 번에 하나의 스레드만 Python 바이트코드를 실행하도록 제한하는 뮤텍스입니다.
CPU-bound 작업에서 멀티스레딩 성능이 제한됩...

[5] NoSQL 데이터베이스는 언제 사용하나요?
    -> 1. 스키마가 자주 변경되는 경우 (유연한 구조)
2. 대량의 비정형 데이터 저장
3. 수평 확장이 필요한 경우
4. 실시간 빅데이터 처리
대표...

-> 실전에서는 이 과정을 수백~수천 번 반복
   시드 5개 -> 50개 -> 500개 -> 5000개로 확장


---
## 2. Evol-Instruct: 기존 데이터를 진화시키기

WizardLM에서 제안한 방식. 기존 instruction을 **점진적으로 복잡하게** 변형한다.

In [6]:
# === Evol-Instruct: 깊이 진화 프롬프트 ===
# 기존 instruction에 제약/복잡도를 추가하여 더 어렵게 만듦

def build_depth_evolve_prompt(original_instruction):
    """기존 instruction을 더 복잡하게 진화시키는 프롬프트"""
    
    strategies = [
        "제약 조건을 추가하세요 (예: 특정 라이브러리 사용 금지, 시간복잡도 제한)",
        "추론 단계를 더 깊게 만드세요 (예: 왜 그런지 설명 요구, 비교 분석)",
        "구체적인 시나리오를 추가하세요 (예: 실제 사용 사례, 에러 상황)",
        "여러 개념을 결합하세요 (예: 두 가지 기술을 함께 사용)",
    ]
    
    strategy = random.choice(strategies)
    
    prompt = f"""다음 instruction을 더 복잡하고 어렵게 진화시키세요.

원본 instruction:
{original_instruction}

진화 전략: {strategy}

규칙:
1. 원본의 주제를 유지하되, 더 깊이 있는 질문으로 변형
2. 실무에서 마주칠 수 있는 현실적인 복잡도 추가
3. 한국어로 작성

진화된 instruction만 출력하세요.
"""
    return prompt, strategy

# 진화 시연
original = "Python에서 리스트의 중복 요소를 제거하는 방법을 설명하세요."
print(f"원본: {original}")
print("\n진화 전략별 예시:")
print("=" * 60)

# 4가지 전략 모두 보여주기
evolved_examples = [
    ("제약 조건 추가",
     "set()을 사용하지 않고 리스트의 중복을 제거하되, 원래 순서를 유지하고 시간복잡도 O(n)을 달성하는 방법을 구현하세요."),
    ("추론 심화",
     "Python에서 리스트 중복 제거 시 set(), dict.fromkeys(), 리스트 컴프리헨션 방식의 시간/공간 복잡도를 비교하고, 각각 어떤 상황에서 최적인지 분석하세요."),
    ("시나리오 추가",
     "100만 개의 딕셔너리 객체를 담은 리스트에서 특정 키 기준으로 중복을 제거해야 합니다. 메모리 제한 1GB 환경에서 효율적인 구현 방법을 설명하세요."),
    ("개념 결합",
     "Python 리스트의 중복 제거와 정렬을 동시에 수행하되, 커스텀 비교 함수를 사용하여 대소문자를 무시하고 중복을 판단하는 코드를 작성하세요."),
]

for strategy, evolved in evolved_examples:
    print(f"\n  [{strategy}]")
    print(f"  -> {evolved}")

원본: Python에서 리스트의 중복 요소를 제거하는 방법을 설명하세요.

진화 전략별 예시:

  [제약 조건 추가]
  -> set()을 사용하지 않고 리스트의 중복을 제거하되, 원래 순서를 유지하고 시간복잡도 O(n)을 달성하는 방법을 구현하세요.

  [추론 심화]
  -> Python에서 리스트 중복 제거 시 set(), dict.fromkeys(), 리스트 컴프리헨션 방식의 시간/공간 복잡도를 비교하고, 각각 어떤 상황에서 최적인지 분석하세요.

  [시나리오 추가]
  -> 100만 개의 딕셔너리 객체를 담은 리스트에서 특정 키 기준으로 중복을 제거해야 합니다. 메모리 제한 1GB 환경에서 효율적인 구현 방법을 설명하세요.

  [개념 결합]
  -> Python 리스트의 중복 제거와 정렬을 동시에 수행하되, 커스텀 비교 함수를 사용하여 대소문자를 무시하고 중복을 판단하는 코드를 작성하세요.


In [7]:
# === Evol-Instruct: 넓이 진화 프롬프트 ===
# 같은 난이도에서 주제를 확장

def build_breadth_evolve_prompt(original_instruction):
    """기존 instruction의 주제를 다른 영역으로 확장"""
    prompt = f"""다음 instruction과 같은 난이도와 형식이지만, 완전히 다른 주제의 instruction을 생성하세요.

원본 instruction:
{original_instruction}

규칙:
1. 난이도와 답변 형식은 유지
2. 프로그래밍/IT 관련이되 다른 세부 주제
3. 한국어로 작성

새로운 instruction만 출력하세요.
"""
    return prompt

# 넓이 진화 예시
breadth_examples = [
    "JavaScript에서 배열의 중복 요소를 제거하는 방법을 설명하세요.",   # 언어만 변경
    "Python에서 딕셔너리의 중복 값을 찾는 방법을 설명하세요.",        # 자료구조 변경
    "SQL에서 테이블의 중복 레코드를 제거하는 방법을 설명하세요.",      # 도메인 변경
]

print(f"원본: {original}")
print(f"\n넓이 진화 (같은 난이도, 다른 주제):")
for ex in breadth_examples:
    print(f"  -> {ex}")

print("\n-> 깊이 진화: 같은 주제를 더 어렵게")
print("   넓이 진화: 같은 난이도로 주제를 넓게")
print("   둘을 조합하면 난이도 × 주제 모두 다양해짐")

원본: Python에서 리스트의 중복 요소를 제거하는 방법을 설명하세요.

넓이 진화 (같은 난이도, 다른 주제):
  -> JavaScript에서 배열의 중복 요소를 제거하는 방법을 설명하세요.
  -> Python에서 딕셔너리의 중복 값을 찾는 방법을 설명하세요.
  -> SQL에서 테이블의 중복 레코드를 제거하는 방법을 설명하세요.

-> 깊이 진화: 같은 주제를 더 어렵게
   넓이 진화: 같은 난이도로 주제를 넓게
   둘을 조합하면 난이도 × 주제 모두 다양해짐


---
## 3. 생성 데이터 품질 검증

In [8]:
# === 합성 데이터 품질 자동 검증 ===
# API로 대량 생성 후 반드시 품질 필터링 필요

def validate_synthetic(item):
    """
    합성 데이터의 기본 품질 검증.
    통과 여부와 사유를 반환.
    """
    issues = []
    
    # 1) 필수 필드 존재
    if not item.get('instruction') or not item.get('output'):
        issues.append("필수 필드 누락")
        return False, issues
    
    # 2) 최소 길이
    if len(item['instruction']) < 10:
        issues.append("instruction 너무 짧음")
    if len(item['output']) < 30:
        issues.append("output 너무 짧음")
    
    # 3) instruction이 질문/지시 형태인지
    has_question = any(w in item['instruction'] for w in ['?', '하세요', '설명', '작성', '알려', '구현'])
    if not has_question:
        issues.append("instruction이 질문/지시 형태가 아님")
    
    # 4) output이 instruction과 관련 있는지 (단어 겹침 확인)
    inst_words = set(item['instruction'].split())
    out_words = set(item['output'].split())
    overlap = len(inst_words & out_words)
    if overlap == 0:
        issues.append("instruction-output 관련성 의심")
    
    # 5) 반복 패턴 탐지
    words = item['output'].split()
    if len(words) > 10:
        # 연속 3단어가 반복되는지 체크
        trigrams = [' '.join(words[i:i+3]) for i in range(len(words)-2)]
        trigram_counts = Counter(trigrams)
        max_repeat = max(trigram_counts.values()) if trigram_counts else 0
        if max_repeat > 3:
            issues.append("output에 반복 패턴")
    
    return len(issues) == 0, issues

# 시뮬레이션된 생성 데이터 + 의도적 불량 데이터 검증
test_data = simulated_generated + [
    {"instruction": "hi", "output": "hello"},  # 너무 짧음
    {"instruction": "This is a statement.", "output": "Some random text " * 20},  # 질문 아님 + 반복
]

print("합성 데이터 품질 검증:")
print("=" * 60)
pass_count = 0
for i, item in enumerate(test_data):
    passed, issues = validate_synthetic(item)
    status = "PASS" if passed else "FAIL"
    if passed:
        pass_count += 1
    inst_preview = item['instruction'][:40]
    print(f"  [{status}] {inst_preview}...")
    if issues:
        for issue in issues:
            print(f"         -> {issue}")

total = len(test_data)
print(f"\n통과율: {pass_count}/{total} ({pass_count/total*100:.0f}%)")

합성 데이터 품질 검증:
  [FAIL] Docker에서 컨테이너와 이미지의 차이를 설명하세요....
         -> instruction-output 관련성 의심
  [FAIL] Python에서 *args와 **kwargs의 차이점은?...
         -> instruction-output 관련성 의심
  [FAIL] TCP와 UDP의 차이점을 설명하세요....
         -> instruction-output 관련성 의심
  [FAIL] Python의 GIL(Global Interpreter Lock)이란 무...
         -> instruction-output 관련성 의심
  [FAIL] NoSQL 데이터베이스는 언제 사용하나요?...
         -> instruction-output 관련성 의심
  [FAIL] hi...
         -> instruction 너무 짧음
         -> output 너무 짧음
         -> instruction이 질문/지시 형태가 아님
         -> instruction-output 관련성 의심
  [FAIL] This is a statement....
         -> instruction이 질문/지시 형태가 아님
         -> instruction-output 관련성 의심
         -> output에 반복 패턴

통과율: 0/7 (0%)


---
## 4. 다양성 분석

In [9]:
# === 생성 데이터의 다양성 분석 ===
# 합성 데이터의 가장 큰 위험: 다양성 부족 (비슷한 내용 반복)

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# 시드 + 생성 데이터 합치기
all_data = seed_data + simulated_generated
all_instructions = [d['instruction'] for d in all_data]

# TF-IDF로 유사도 계산
vectorizer = TfidfVectorizer(max_features=1000)
tfidf = vectorizer.fit_transform(all_instructions)
sim = cosine_similarity(tfidf)

# 평균 유사도 (자기 자신 제외)
n = len(all_instructions)
mask = np.ones((n, n), dtype=bool)
np.fill_diagonal(mask, False)
avg_sim = sim[mask].mean()

# 시드 내 유사도 vs 생성 데이터 내 유사도
seed_n = len(seed_data)
seed_sim = sim[:seed_n, :seed_n][np.triu_indices(seed_n, k=1)].mean()
gen_sim = sim[seed_n:, seed_n:][np.triu_indices(len(simulated_generated), k=1)].mean()

print("다양성 분석:")
print("=" * 60)
print(f"  전체 평균 유사도: {avg_sim:.3f}")
print(f"  시드 내부 유사도: {seed_sim:.3f}")
print(f"  생성 데이터 내부 유사도: {gen_sim:.3f}")

print("\n판단 기준:")
print("  0.3 이하: 다양성 좋음")
print("  0.3~0.5: 보통")
print("  0.5 이상: 다양성 부족, 시드 다양화 필요")

if gen_sim > 0.5:
    print("\n경고: 생성 데이터 유사도가 높습니다!")
    print("  -> 시드 데이터를 더 다양하게 만들거나")
    print("  -> 프롬프트에 '이전 생성 결과와 다른 주제' 조건 추가")
else:
    print("\n다양성 양호!")

다양성 분석:
  전체 평균 유사도: 0.038
  시드 내부 유사도: 0.030
  생성 데이터 내부 유사도: 0.008

판단 기준:
  0.3 이하: 다양성 좋음
  0.3~0.5: 보통
  0.5 이상: 다양성 부족, 시드 다양화 필요

다양성 양호!


---
## 5. 실전 생성 파이프라인 설계

In [10]:
# === 실전 합성 데이터 파이프라인 전체 구조 ===
# 실제 API 호출 부분은 주석으로 표시

class SyntheticDataPipeline:
    """
    합성 데이터 생성 파이프라인.
    실전에서는 generate_with_llm()에 실제 API 호출을 구현.
    """
    
    def __init__(self, seed_data):
        self.seed_data = seed_data
        self.generated = []
        self.rejected = []
    
    def generate_with_llm(self, prompt):
        """
        실제 API 호출 자리.
        실전에서는 여기에 OpenAI/Claude API를 연결.
        
        예시:
        response = client.chat.completions.create(
            model='gpt-4',
            messages=[{'role': 'user', 'content': prompt}]
        )
        return json.loads(response.choices[0].message.content)
        """
        # 시뮬레이션: 실제로는 LLM 응답
        return None
    
    def self_instruct_round(self, num_generate=5):
        """Self-Instruct 한 라운드 실행"""
        prompt = build_self_instruct_prompt(self.seed_data, num_generate)
        results = self.generate_with_llm(prompt)
        
        if results:
            for item in results:
                passed, issues = validate_synthetic(item)
                if passed:
                    self.generated.append(item)
                else:
                    self.rejected.append((item, issues))
    
    def evolve_round(self, data_to_evolve):
        """Evol-Instruct 한 라운드 실행"""
        for item in data_to_evolve:
            prompt, strategy = build_depth_evolve_prompt(item['instruction'])
            result = self.generate_with_llm(prompt)
            
            if result:
                passed, issues = validate_synthetic(result)
                if passed:
                    self.generated.append(result)
    
    def summary(self):
        """파이프라인 실행 결과 요약"""
        total_attempts = len(self.generated) + len(self.rejected)
        if total_attempts == 0:
            print("아직 생성된 데이터가 없습니다 (시뮬레이션 모드)")
            return
        pass_rate = len(self.generated) / total_attempts * 100
        print(f"생성 결과:")
        print(f"  시도: {total_attempts}개")
        print(f"  통과: {len(self.generated)}개")
        print(f"  탈락: {len(self.rejected)}개")
        print(f"  통과율: {pass_rate:.1f}%")

# 파이프라인 구조 확인
pipeline = SyntheticDataPipeline(seed_data)

print("합성 데이터 파이프라인 구조:")
print("=" * 60)
print("  1. 시드 데이터 준비 (5~10개 직접 작성)")
print("  2. Self-Instruct: 시드에서 새 데이터 생성")
print("  3. Evol-Instruct: 기존 데이터를 복잡하게 진화")
print("  4. 품질 검증: 자동 필터 + LLM-as-Judge")
print("  5. 다양성 분석: TF-IDF 유사도 체크")
print("  6. 반복: 충분한 양이 될 때까지 2-5 반복")
print("\n실전 팁:")
print("  - 시드 데이터가 고품질이어야 함 (쓰레기 in = 쓰레기 out)")
print("  - 실제 데이터 : 합성 데이터 = 7 : 3 비율 권장")
print("  - 합성 데이터만으로는 Model Collapse 위험")

합성 데이터 파이프라인 구조:
  1. 시드 데이터 준비 (5~10개 직접 작성)
  2. Self-Instruct: 시드에서 새 데이터 생성
  3. Evol-Instruct: 기존 데이터를 복잡하게 진화
  4. 품질 검증: 자동 필터 + LLM-as-Judge
  5. 다양성 분석: TF-IDF 유사도 체크
  6. 반복: 충분한 양이 될 때까지 2-5 반복

실전 팁:
  - 시드 데이터가 고품질이어야 함 (쓰레기 in = 쓰레기 out)
  - 실제 데이터 : 합성 데이터 = 7 : 3 비율 권장
  - 합성 데이터만으로는 Model Collapse 위험


---
## 정리

| 기법 | 핵심 | 적합한 경우 |
|------|------|------------|
| **Self-Instruct** | 시드에서 새로 생성 | 데이터가 아예 없을 때 |
| **Evol-Instruct (깊이)** | 기존 데이터를 더 어렵게 | 난이도 다양화 |
| **Evol-Instruct (넓이)** | 다른 주제로 확장 | 주제 다양화 |
| **품질 검증** | 자동 필터 + 사람 검수 | 항상 필수 |
| **다양성 분석** | TF-IDF 유사도 체크 | 생성 후 반드시 |